In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from scipy.stats import randint

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, precision_score

# Preprocessing Pipeline Plan

- Limit observations to only the first encounter
- Drop additional prescribed medication due to limited clinical relevance
- Conduct feature decomposition on insulin and metformin to include a binary flag for being prescribed the medication and an ordinal encoding for the change
- Create train, validation and test splits
- Preprocess features
    - Convert data types (admission source, admission type)
    - Scale numeric features
    - Encode categoricals (OHE and ordinal)
    - Encode target (multi-class, binary)

## Experimenting with Feature Decomposition

Decomposing the insulin and metformin columns into two each: one binary flag to indicate if someone is on the drug at all, another ordinal encoding encoding if the patient is on the drug to indicate if there was a change in dosing.

In [ ]:
features = pd.read_parquet("data/diabetes_features.parquet")
target = pd.read_parquet("data/diabetes_target.parquet")

### Limiting the Data to the First Encounter Only

In [ ]:
# Creating a dataframe for selecting first encounters
splitting_df = features.copy()
splitting_df['target'] = (target['readmitted'] == "<30").astype(int)

# Keeping only the first encounter per patient_nbr
split_df = splitting_df.drop_duplicates('patient_nbr', keep='first')

### Preparing Features

Binary and Ordinal Splits for Insulin and Metformin

In [ ]:
# Creating a feature to indicate if a patient is not on insulin or metformin
split_df['missing_insulin'] = (split_df.insulin == "No").astype(int)
split_df['missing_metformin'] = (split_df.metformin == "No").astype(int)

# Creating a dictionary to encode changes in insulin and metformin
encode_dict = {'No': 0,
               "Steady": 0,
               "Up": 1,
               "Down": -1}

# Applying the encoding to engineer new features
split_df['metformin_change'] = split_df['metformin'].apply(lambda x: encode_dict[x])
split_df['insulin_change'] = split_df['insulin'].apply(lambda x: encode_dict[x])

Binary missingness encoding for weight

In [ ]:
split_df['missing_weight'] = (split_df['weight'] == "?").astype("int")

Binary indicator for missingness to one-hot encode:
- medical_specialty
- payer_code

In [ ]:
# Utility function for encoding missing values in a column for one-hot encoding
def missing_cleaner(x, missing_code, encoding=""):
    if x == missing_code:
        return encoding
    else:
        return x

split_df['payer_code_cleaned'] = split_df['payer_code'].apply(missing_cleaner, args=("?", "missing_payer"))
split_df['medical_specialty_cleaned'] = split_df['medical_specialty'].apply(missing_cleaner, args=("?", "missing_medical_specialty"))

Binary indicator and ordinal encoding for A1C and max_glu_serum

In [ ]:
split_df['missing_a1c'] = (split_df['A1Cresult'] == "None").astype("int")
split_df['missing_max_glu_serum'] = (split_df['max_glu_serum'] == "None").astype("int")

#### Creating Train and Test Splits

In [ ]:
# List of features to drop
drop_features = ['patient_nbr', 'encounter_id', 'target',
                 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide',
                 'glyburide', 'tolbutamide',
                 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
                 'tolazamide', 'examide', 'citoglipton', 'insulin',
                 'glyburide.metformin', 'glipizide.metformin',
                 'glimepiride.pioglitazone', 'metformin.rosiglitazone',
                 'metformin.pioglitazone', 'metformin', 'insulin',
                 'A1Cresult', 'weight', 'diag_1', 'diag_2', 'diag_3',
                 'max_glu_serum', 'change']

# Building X & y dataframes
y = split_df['target']
X = split_df.drop(columns=drop_features)

# Creating train and test splits
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

## Setting up the Pipeline

Features for scaling:
- time_in_hospital
- num_lab_procedures
- num_procedures
- num_medications
- number_outpatient
- number_inpatient
- number_diagnoses

Features for OHE:
- race
- gender
- admission_type_id
- dicharge_disposition_id
- admission_source_id
- payer_code
- medical_specialty

Features for ordinal encoding:
- Age
- Metformin (already encoded)
- Insulin (already encoded)

#### OHE Columns

In [ ]:
OHEFEATURES = ['race', 'gender', 'admission_type_id',
               'discharge_disposition_id', 'admission_source_id',
               'payer_code', 'medical_specialty']

# Pipeline for OHE
ohe_pipeline = Pipeline([('handle_outliers', SimpleImputer(strategy='most_frequent')),
                         ('encoding', OneHotEncoder(handle_unknown='ignore',
                                                    min_frequency=0.01,
                                                    sparse_output=False
                                                    ))])

#### Columns for Scaling

In [ ]:
SCALING_FEATURES = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                    'num_medications', 'number_outpatient', 'number_inpatient', 'number_diagnoses']

# Pipeline for Scaling
scaling_pipeline = Pipeline([('scaling', MinMaxScaler())])

#### Columns for Ordinal Encoding

In [ ]:
# Setting up a list of columns for ordinal encoding
ORDINAL_COLS = ['age']

# Creating a list of categories for the age column
age_categories = sorted(split_df['age'].unique())

# Setting up a pipeline for ordinal encoding
ordinal_pipeline = Pipeline([('ordinal_encoding', OrdinalEncoder(categories=[age_categories]))])

#### Setting up the Column Transformer

In [ ]:
ct = ColumnTransformer([('OHE', ohe_pipeline, OHEFEATURES),
                        ('scaling', scaling_pipeline, SCALING_FEATURES),
                        ('ordinal_encoding', ordinal_pipeline, ORDINAL_COLS)])

# Setting the output to return a pandas dataset
ct.set_output(transform='pandas')

## Fitting the Model

In [ ]:
# Setting up a dictionary of models
models = {'random_forest': RandomForestClassifier(class_weight='balanced'),
          'log_reg': LogisticRegression()}

# Creating the parameter grid
param_grid = {
    'random_forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10],
        'model__min_samples_split': [2, 5]
    },
    'log_reg': {
        'model__C': np.logspace(-3, 2, 6)
    }
}

# Setting up a scoring dictionary
scoring_dict = {
    'average_precision': 'average_precision',
    'recall': 'recall',
    'roc_auc': 'roc_auc',
    'precision': make_scorer(precision_score, zero_division=0)
    }

In [ ]:
if False:
    # Setting up a StratifiedKFold for cross-validation
    cv = StratifiedKFold(n_splits=5,
                        random_state=42,
                        shuffle=True)

    # Creating a dictionary for CV results
    cv_results = {}


    # Setting up the loop to evaluate the different models across the parameter grids
    for model_name, model in models.items():

        # Pipeline for model training
        model_training_pipeline = Pipeline([('preprocessing', ct),
                                            ('model', model)])

        # Setting up the grid for CV
        grid = GridSearchCV(model_training_pipeline,
                            param_grid=param_grid[model_name],
                            scoring=scoring_dict,
                            refit='average_precision',
                            cv=cv)

        # Storing the results
        cv_results[model_name] = grid.fit(X_train, y_train)

In [ ]:
if False:    
    results_df = pd.concat(
        {name: pd.DataFrame(grid.cv_results_) for name, grid in cv_results.items()},
        names=['model']
    ).reset_index(level='model')

    # The columns you actually care about:
    cols = ['model', 'params', 'mean_test_average_precision',
            'mean_test_recall', 'mean_test_roc_auc', 'rank_test_average_precision']
    results_df[cols].sort_values('mean_test_average_precision', ascending=False)

#### CV Findings:

While Logistic regression produces similar results for average precision, its recall is extraoridarily low. This is not acceptable since recall measures how well the model detects patients that are likely to readmit.

Performance of Random Forest on average precision is best with deeper trees and a smaller number of samples for a split. However, these deep trees have a lower recall than shallow trees, suggesting the model is overfitting. Improved recall with a lower number of samples suggests the model may be learning edge cases for a smaller number of patients, or is subject overfitting.

## Model Refinement

Hyperparameter tuning and feature selection

#### Random Grid Search

In [ ]:
if False: 
    # Setting up a new pipeline for hyperparameter tuning
    hyperparameter_pipe = Pipeline([('preprocessing', ct),
                                    ('model', RandomForestClassifier(
                                        random_state=42,
                                        class_weight='balanced'
                                        ))])

    # Creating a randomized parameter grid
    rand_hyper_grid = {'model__max_depth': randint(2, 15),
                'model__min_samples_split': randint(2, 10),
                'model__n_estimators': randint(50, 300),
                'model__max_features': ['sqrt', 'log2', 0.3, 0.5]}

    # Setting up a random grid search object
    random_cv = RandomizedSearchCV(estimator=hyperparameter_pipe,
                                cv=5,
                                param_distributions=rand_hyper_grid,
                                n_iter=20,
                                scoring=scoring_dict,
                                random_state=42,
                                refit='recall')

    # Fitting the random grid
    random_cv.fit(X_train, y_train)

In [ ]:
if False:
    # Dataframe for CV results
    rand_grid_cv_results = pd.DataFrame(random_cv.cv_results_)

    rand_grid_cv_results.sort_values('mean_test_recall', ascending=False).head(10)

In [ ]:
if False:
    sns.relplot(data=rand_grid_cv_results,
                x='mean_test_recall',
                y='mean_test_average_precision',
                hue='param_model__min_samples_split',
                style='param_model__max_features',
                size='param_model__n_estimators')
    plt.title("Average Precision vs Recall for Randomized Search CV")
    plt.show()

In [ ]:
if False:
    sns.relplot(data=rand_grid_cv_results,
                x='mean_test_recall',
                y='mean_test_average_precision',
                hue='param_model__max_depth',
                style='param_model__max_features',
                size='param_model__n_estimators')
    plt.title("Average Precision vs Recall for Randomized Search CV")
    plt.show()

#### Random CV Results

- Max depth has a clear impact on model performance: deep trees perform the worst while very shallow trees overfit and precision suffers
- A moderate numbe =r of estimators (~200) provides great recall without a large penalty on precision
- The model performs best with a larger number of parameters (~0.5)
- A high number (~9) of minimum samples per split performs best

#### Regular Grid Search

Searching the smaller space of hyperparameters based on the results from random CV

In [ ]:
# Creating a refined hyperparameter grid based on the results of the random search
refined_hyper_grid = {
    'model__max_depth': [5, 6, 7],
    'model__min_samples_split': [8, 9, 10],
    'model__n_estimators': [175, 200, 225],
    'model__max_features': [0.4, 0.5, 0.6]
}

# Running the tune
refined_cv = GridSearchCV(hyperparameter_pipe,
                         cv=5,
                         param_grid=refined_hyper_grid,
                         scoring=scoring_dict,
                         refit='recall',
                         n_jobs=-1)

refined_cv.fit(X_train, y_train)

In [ ]:
refined_cv_results = pd.DataFrame(refined_cv.cv_results_)
refined_cv_results.to_excel("experiments/refined_cv_results.xlsx", index=False)

In [ ]:
refined_cv_results.sort_values('mean_test_recall', ascending=False).head(10)

In [ ]:
sns.relplot(data=refined_cv_results,
            x='mean_test_recall',
            y='mean_test_average_precision',
            hue='param_model__min_samples_split',
            style='param_model__max_features',
            size='param_model__n_estimators')
plt.title("Average Precision vs Recall for Randomized Search CV")
plt.ylim(0.15, 0.2)
plt.show()

In [ ]:
sns.relplot(data=refined_cv_results,
            x='mean_test_recall',
            y='mean_test_average_precision',
            hue='param_model__max_depth',
            style='param_model__max_features',
            size='param_model__n_estimators')
plt.title("Average Precision vs Recall for Randomized Search CV")
plt.ylim(0.16, 0.18)
plt.show()

#### Refitting the Optimal Model

Optimal Parameters:
max_features = 0.6
n_estimators = 200
max_depth = 5
min_samples_split = 10

In [ ]:
# Setting up a dict of optimal hyperparameters
optimal_hypers = {
    'max_depth': 5,
    'min_samples_split': 10,
    'n_estimators': 200,
    'max_features': 0.6,
    'class_weight': 'balanced'
}

optimal_rf = RandomForestClassifier(n_jobs=-1,
                                    **optimal_hypers)

optimal_pipeline = Pipeline([('preprocessor', ct),
                             ('model', optimal_rf)])

optimal_pipeline.fit(X_train, y_train)

#### Feature Importance

In [ ]:
explainer = shap.TreeExplainer(optimal_pipeline['model'])
transformed_x_test = optimal_pipeline[:-1].transform(X_test)
shap_values = explainer.shap_values(transformed_x_test)

In [ ]:
shap.summary_plot(shap_values[:, :, 1], transformed_x_test)